# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Method: Gradient Boosting Classifier (sklearn)

Why: Lane 2 (Content Opportunity Scoring) has an imbalanced target — only a small
fraction of pages are flagged. GBM captures non-linear feature interactions (e.g.
combined effect of CTR + impressions) that my Week-4 rule-based baseline could not.
The dataset size (~30K rows) is reasonable for this method, and it also provides
feature importance, which is used in the interpretation section below.

In [14]:
!pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

from getpass import getpass
hf_token = getpass("HF token daalo: ")

import duckdb
con = duckdb.connect()

# SAHI syntax - dataset card se
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

HF token daalo: ··········


In [16]:
files = con.execute(f"""
    SELECT * FROM glob('{rel}/fact_content_daily_performance/**')
""").df()

import pandas as pd
pd.set_option('display.max_colwidth', None)
print(files.to_string())



                                                                                                      file
0   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
1   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
2   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
3   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
4   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
5   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
6   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
7   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08/data_0.parquet
8   hf://datasets/FlyRank/internship-

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
Split: Stratified train/test split (80/20), stratified on the target column.

Why: Client/domain grouping is not directly available in the FlyRank warehouse data
for this lane, so the risk of group/time leakage is low here. Stratification is used
because the target is a rare class — a plain random split could leave too few flagged
examples in the test set, making evaluation unreliable.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score
import pandas as pd

feature_cols = ['clicks_delta_pct', 'position_delta', 'ctr_b', 'ctr_a',
                 'search_volume', 'competition_level', 'word_count']

features_df['competition_level'] = features_df['competition_level'].astype('category').cat.codes

X = features_df[feature_cols]
y = features_df['will_decline']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
Observed: F1 changed from [baseline value] (Week-4 baseline) to [new value]
(Gradient Boosting) on the same test split. [One sentence on whether this is an
improvement and by how much.]


In [ ]:
model = GradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(classification_report(y_test, preds))

baseline_preds = (features_df.loc[X_test.index, 'status'] == 'declining').astype(int)
model_f1 = f1_score(y_test, preds)
baseline_f1 = f1_score(y_test, baseline_preds)

results = pd.DataFrame({
    'Model': ['Rule-Based Baseline (Week 4)', 'Gradient Boosting'],
    'F1': [baseline_f1, model_f1]
})
print(results)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
Observed: CTR is the most dominant feature, consistent with the pattern seen in the
Week-4 baseline. The model produces more false negatives on pages with low impressions
but high CTR — the signal appears directionally weaker at low data volume. This is
useful as decision-support, but flagging should not be fully automated on this signal alone.


In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print(importance_df)

import matplotlib.pyplot as plt
plt.barh(importance_df['feature'], importance_df['importance'])
plt.title('Feature Importance — What Predicts Decline')
plt.savefig('feature_importance.png')
plt.show()

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.